In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_1.py ──
"""
Shared infrastructure for MLFP04 Exercise 1 — Clustering Zoo.

Contains: customer-feature loading & standardisation, metric helpers,
subsampling utilities, and output-directory management.

Technique-specific code (K-means elbow, linkage methods, DBSCAN epsilon
search, HDBSCAN, spectral Laplacian, AutoMLEngine) does NOT belong
here — it lives in the per-technique files under modules/mlfp04/solutions/ex_1/.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex1_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared random state so every technique file is reproducible
RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════


def load_customers() -> tuple[pl.DataFrame, list[str]]:
    """Load the e-commerce customer dataset and return (df, numeric_feature_cols).

    The dataset (from MLFP03) is ~6K rows of Singapore e-commerce customers
    with recency, frequency, monetary, basket-size, and channel features.
    Clustering is unsupervised segmentation: no labels, just behaviour.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    return customers.drop_nulls(subset=feature_cols), feature_cols


def standardise(
    df: pl.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, StandardScaler]:
    """Return (X_scaled, scaler). Zero mean, unit variance — mandatory for
    all distance-based clustering."""
    X, _, _ = to_sklearn_input(df, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — spectral / hierarchical are O(n^2) or worse
# ════════════════════════════════════════════════════════════════════════


def subsample(
    X: np.ndarray, n: int, seed: int = RANDOM_STATE
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_sub, idx) where idx are the original row indices chosen."""
    rng = np.random.default_rng(seed)
    n = min(n, X.shape[0])
    idx = rng.choice(X.shape[0], n, replace=False)
    return X[idx], idx


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def score_partition(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Compute silhouette, Calinski-Harabasz, Davies-Bouldin.

    Points with label == -1 (DBSCAN noise) are excluded. Returns NaN
    fields if fewer than 2 valid clusters remain.
    """
    valid = labels != -1
    labs = labels[valid]
    data = X[valid]
    if data.shape[0] < 2 or len(set(labs.tolist())) < 2:
        return {
            "n_clusters": len(set(labs.tolist())),
            "silhouette": float("nan"),
            "calinski_harabasz": float("nan"),
            "davies_bouldin": float("nan"),
        }
    return {
        "n_clusters": len(set(labs.tolist())),
        "silhouette": float(silhouette_score(data, labs)),
        "calinski_harabasz": float(calinski_harabasz_score(data, labs)),
        "davies_bouldin": float(davies_bouldin_score(data, labs)),
    }


def agreement(labels_a: np.ndarray, labels_b: np.ndarray) -> dict[str, float]:
    """ARI and NMI between two label vectors, ignoring noise (-1)."""
    valid = (labels_a >= 0) & (labels_b >= 0)
    if valid.sum() < 2:
        return {"ari": float("nan"), "nmi": float("nan")}
    return {
        "ari": float(adjusted_rand_score(labels_a[valid], labels_b[valid])),
        "nmi": float(normalized_mutual_info_score(labels_a[valid], labels_b[valid])),
    }


def print_metric_row(name: str, m: dict[str, Any]) -> None:
    """One-line summary of a partition's metrics."""
    print(
        f"  {name:<14} K={m['n_clusters']:>3}  "
        f"sil={m['silhouette']:>7.4f}  "
        f"CH={m['calinski_harabasz']:>9.0f}  "
        f"DB={m['davies_bouldin']:>7.4f}"
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION OUTPUT PATH
# ════════════════════════════════════════════════════════════════════════


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 1.3: Density-Based Clustering (DBSCAN + HDBSCAN)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Apply DBSCAN with core/border/noise classification and select
#     epsilon via a k-distance plot
#   - Apply HDBSCAN to skip epsilon altogether by building a hierarchy of
#     DBSCAN clusterings and extracting the most stable partition
#   - Decide when "noise" is a feature, not a bug
#
# PREREQUISITES: 01_kmeans.py (feature setup).
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — density-based clustering and the epsilon/minPts contract
#   2. Build — k-distance plot and DBSCAN sweep across epsilon
#   3. Train — HDBSCAN with eom vs leaf cluster selection
#   4. Visualise — k-distance elbow plot
#   5. Apply — Singapore Grab ride-hail hotspot discovery, $ impact
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from dotenv import load_dotenv
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from kailash_ml import ModelVisualizer


load_dotenv()

try:
    import hdbscan as hdbscan_lib
except ImportError:
    hdbscan_lib = None



## THEORY — Density as the Cluster Definition


In [ ]:
# K-means and hierarchical cluster by MINIMISING a distance objective.
# DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
# redefines a cluster as "a region of high point density, separated from
# other regions by areas of low density". Two hyperparameters:
#
#   epsilon  - neighbourhood radius in input space
#   minPts   - minimum points inside that radius to be "core"
#
# Every point is classified as:
#   Core   - has ≥ minPts neighbours within epsilon
#   Border - not core, but within epsilon of a core point
#   Noise  - neither (label = -1)
#
# Two consequences:
#   1. Clusters can be any SHAPE (moons, rings, arbitrary blobs)
#   2. Points in sparse regions get label -1 ("noise") instead of being
#      forced into a cluster they do not belong to
#
# HDBSCAN fixes DBSCAN's biggest weakness: epsilon is hard to choose and
# changes with the data. HDBSCAN runs DBSCAN for EVERY epsilon at once,
# builds a hierarchy, and extracts the clusters that stay stable across
# the widest range of densities. You specify only min_cluster_size.



## TASK 2 — BUILD: Load data, run k-distance plot, sweep DBSCAN epsilon


In [ ]:
customers, feature_cols = load_customers()
X_scaled, _ = standardise(customers, feature_cols)
n_samples = X_scaled.shape[0]

print("=" * 70)
print("  Density-Based Clustering on Singapore E-commerce Customers")
print("=" * 70)
print(f"  Samples: {n_samples:,}  features: {X_scaled.shape[1]}")

# k-distance plot: the canonical epsilon-selection tool for DBSCAN
K_NN = 10  # also used as minPts
nn = NearestNeighbors(n_neighbors=K_NN)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_dist = np.sort(distances[:, -1])

# Approximate the elbow via maximum second-derivative
diffs2 = np.diff(np.diff(k_dist))
elbow_idx = int(np.argmax(diffs2)) + 2
eps_suggested = float(k_dist[elbow_idx])

print(f"\n  k-distance elbow at eps ≈ {eps_suggested:.4f}")
print(f"  Range: [{k_dist[0]:.4f}, {k_dist[-1]:.4f}]")

# DBSCAN sweep around the suggested epsilon
print(f"\n  {'eps':>8} {'Clusters':>10} {'Noise %':>10} {'Silhouette':>12}")
print("  " + "─" * 44)
dbscan_results: dict[float, dict] = {}
for eps in (eps_suggested * 0.7, eps_suggested, eps_suggested * 1.3):
    labels = DBSCAN(eps=eps, min_samples=K_NN, n_jobs=-1).fit_predict(X_scaled)
    k = len(set(labels.tolist())) - (1 if -1 in labels else 0)
    noise_pct = float((labels == -1).mean())
    valid = labels != -1
    sil = (
        silhouette_score(X_scaled[valid], labels[valid])
        if valid.sum() >= 2 and k >= 2
        else float("nan")
    )
    dbscan_results[eps] = {"labels": labels, "k": k, "noise_pct": noise_pct, "sil": sil}
    print(f"  {eps:>8.4f} {k:>10} {noise_pct:>9.1%} {sil:>12.4f}")

db_labels = dbscan_results[eps_suggested]["labels"]
n_noise = int((db_labels == -1).sum())
print(f"\n  DBSCAN (eps={eps_suggested:.4f}, minPts={K_NN}):")
print(f"    Non-noise points: {n_samples - n_noise:,}")
print(f"    Noise points:     {n_noise:,} ({n_noise/n_samples:.1%})")
print(f"    Clusters found:   {dbscan_results[eps_suggested]['k']}")



### Checkpoint 1


In [ ]:
assert eps_suggested > 0, "Task 2: suggested epsilon should be positive"
assert any(
    r["k"] >= 2 for r in dbscan_results.values()
), "Task 2: DBSCAN should find at least 2 clusters for one eps"
print("\n  [ok] Checkpoint 1 passed — DBSCAN epsilon selection complete\n")



## TASK 3 — TRAIN: HDBSCAN with eom vs leaf cluster selection


In [ ]:
if hdbscan_lib is None:
    raise ImportError(
        "hdbscan package is required for this exercise. "
        "Install with: uv add hdbscan  (or  pip install hdbscan)"
    )

hdb_eom = hdbscan_lib.HDBSCAN(
    min_cluster_size=50, min_samples=10, cluster_selection_method="eom"
)
hdb_eom_labels = hdb_eom.fit_predict(X_scaled)
n_eom = len(set(hdb_eom_labels.tolist())) - (1 if -1 in hdb_eom_labels else 0)

hdb_leaf = hdbscan_lib.HDBSCAN(
    min_cluster_size=50, min_samples=10, cluster_selection_method="leaf"
)
hdb_leaf_labels = hdb_leaf.fit_predict(X_scaled)
n_leaf = len(set(hdb_leaf_labels.tolist())) - (1 if -1 in hdb_leaf_labels else 0)

noise_eom = float((hdb_eom_labels == -1).mean())
valid_eom = hdb_eom_labels != -1
sil_eom = (
    silhouette_score(X_scaled[valid_eom], hdb_eom_labels[valid_eom])
    if valid_eom.sum() >= 2 and n_eom >= 2
    else float("nan")
)

print(f"  HDBSCAN cluster-selection comparison:")
print(
    f"    EOM (Excess of Mass) : {n_eom} clusters  noise={noise_eom:.1%}  sil={sil_eom:.4f}"
)
print(f"    Leaf                 : {n_leaf} clusters  (finest granularity)")
print("  EOM is the production default — it picks the most stable clusters")
print("  across all density levels. Leaf gives the finest partition and is")
print("  useful when you explicitly want small sub-clusters.")



### Checkpoint 2


In [ ]:
assert n_eom >= 1, "Task 3: HDBSCAN EOM should find at least 1 cluster"
assert 0 <= noise_eom <= 1, "Task 3: noise fraction must be in [0, 1]"
print("\n  [ok] Checkpoint 2 passed — HDBSCAN auto-discovers clusters\n")



## TASK 4 — VISUALISE: the k-distance elbow that drove epsilon selection


In [ ]:
viz = ModelVisualizer()
fig = viz.training_history(
    {"k-distance (sorted)": k_dist.tolist()},
    x_label="Point index (sorted)",
)
fig.update_layout(
    title=f"DBSCAN k-distance plot (k={K_NN})  elbow at eps≈{eps_suggested:.4f}"
)
fig.write_html(str(out_path("03_dbscan_k_distance.html")))
print(f"  Saved: {out_path('03_dbscan_k_distance.html')}")
print("  Read the plot: y-axis is the distance from each point to its k-th")
print("  nearest neighbour, sorted. The 'elbow' (max curvature) is the")
print("  epsilon where density transitions from 'cluster' to 'noise'.")



### Checkpoint 3


In [ ]:
assert out_path("03_dbscan_k_distance.html").exists(), "Task 4: visualisation not saved"
print("\n  [ok] Checkpoint 3 passed — k-distance visualisation rendered\n")



## TASK 5 — APPLY: Grab Singapore Ride-Hail Hotspot Discovery


In [ ]:
# SCENARIO: Grab Singapore dispatches 400,000+ rides/day. The ops team
# needs to position driver incentives near "hotspots" — dense pickup
# clusters that form and dissolve throughout the day (CBD 8am, Marina
# Bay Sands 11pm, Changi T1 arrivals hall 06:00). K-means would FORCE a
# partition of every pickup in the city, including sparse residential
# areas. Hierarchical would collapse under 400K points.
#
# Why HDBSCAN is the right tool here:
#   - Hotspots have arbitrary SHAPES (Orchard Rd is a long thin strip)
#   - True sparse regions should stay sparse ("noise" is correct for a
#     single lonely pickup in Punggol at 3am — do not build a cluster)
#   - Density VARIES by neighbourhood (CBD is dense; Tuas is sparse) —
#     DBSCAN's single epsilon fails here; HDBSCAN's multi-density
#     hierarchy handles it natively
#   - min_cluster_size maps directly to a business rule: "ignore any
#     hotspot with fewer than N trips/hour"
#
# BUSINESS IMPACT: Grab SEA's 2024 annual report discloses ~US$2.4B in
# mobility gross transaction value for Singapore. Driver-incentive
# mis-targeting (paying surge for areas where demand already exists or
# missing genuine under-served areas) burns an estimated 3-5% of the
# incentive budget. Incentives are ~8% of mobility GTV (~US$192M/year).
# HDBSCAN-based hotspot discovery conservatively recovers ~20% of waste:
#     US$192M × 0.04 average waste × 0.20 recovery = US$1.54M / year
# (roughly S$2.05M/year) — and more importantly, driver satisfaction
# improves because surge lands where actual demand is, not where a
# K-means centroid happened to fall.

print("  APPLY — Grab SG Hotspot Discovery")
print("  ─────────────────────────────────────────────────────────────────")
for cid in sorted(set(int(c) for c in hdb_eom_labels.tolist() if c >= 0)):
    n = int((hdb_eom_labels == cid).sum())
    print(f"    Hotspot {cid}: {n:>5,} customers ({n/n_samples:6.1%})")
print(
    f"    Noise (truly sparse): {int((hdb_eom_labels == -1).sum()):,} customers "
    f"({float((hdb_eom_labels == -1).mean()):6.1%})"
)
print("    Estimated annual incentive waste recovery: S$2.05M")



### Checkpoint 4


In [ ]:
assert (
    int((hdb_eom_labels >= 0).sum() + (hdb_eom_labels == -1).sum()) == n_samples
), "Task 5: HDBSCAN labels must cover every sample"
print("\n  [ok] Checkpoint 4 passed — hotspot partition valid\n")



## REFLECTION


[x] DBSCAN defines clusters by local density, not distance to a centroid
  [x] epsilon is chosen via the k-distance elbow, minPts via domain rule
  [x] Noise (label = -1) is a FEATURE: sparse points stay unassigned
  [x] HDBSCAN eliminates epsilon by running DBSCAN at every density and
      extracting the most persistent clusters
  [x] eom (Excess of Mass) vs leaf cluster selection: eom is default;
      leaf is for finest granularity
  [x] Mapped the method onto Grab SG hotspot discovery — S$2.05M/year
      recovered driver-incentive budget

  KEY INSIGHT: If your data has VARIABLE density (CBD vs suburbs) or
  arbitrary cluster SHAPES (strips, rings, moons), force-fitting K-means
  will give you nonsense. Density-based clustering is what you reach for.

  Next: 04_spectral.py — when you know the clusters are non-convex and
  you need the graph structure to find them.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

